###Creating the Medallion Architecture Schemas

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.quarantine
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

print("Medallion schemas created successfully.")

In [0]:
spark.sql("""
SHOW SCHEMAS IN workspace
""").show()

### Ingesting the Customer Data into Bronze

**Setting the Path**


In [0]:
from pyspark.sql.functions import *
SOURCE_PATH=(
    "s3://rahul-company-merger-data-lake/"
    "company_a/raw/customers"
)

SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_a/customers/"
)

CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_a/customers/"
)

TARGET_TABLE = (
    "workspace.bronze.company_a_customers"
)

**Reading using Auto Loader**

In [0]:
customers_raw_df=(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv",)
    .option("header","true",)
    .option("cloudFiles.schemaLocation",SCHEMA_PATH,)
    .option("cloudFiles.inferColumnTypes","false",)
    .load(SOURCE_PATH)
)


**Adding the metadata**

In [0]:
customers_bronze_df = (
    customers_raw_df
    .withColumn(
        "_source_system",
        lit("company_a"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Customer Data to Bronze Delta table**


In [0]:
query = (
    customers_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        TARGET_TABLE
    )
)

query.awaitTermination()

print(
    "Company A customers Bronze ingestion completed."
)

**Validate the bronze table**


In [0]:
display(
    spark.table(
        "workspace.bronze.company_a_customers"
    )
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_customer_count
FROM workspace.bronze.company_a_customers
""").show()

In [0]:
spark.sql("""
SELECT
    _source_system,
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_a_customers
GROUP BY
    _source_system,
    _source_file
""").show(truncate=False)

### Ingesting Products data into Bronze

**Setting the path**


In [0]:
PRODUCTS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_a/raw/products/"
)

PRODUCTS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_a/products/"
)

PRODUCTS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_a/products/"
)

PRODUCTS_TARGET_TABLE = (
    "workspace.bronze.company_a_products"
)

**Reading using Autoloader**

In [0]:
products_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        PRODUCTS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(PRODUCTS_SOURCE_PATH)
)

**Adding the metadata**

In [0]:
products_bronze_df = (
    products_raw_df
    .withColumn(
        "_source_system",
        lit("company_a"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Products Data to Bronze Delta table**

In [0]:
products_query = (
    products_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        PRODUCTS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        PRODUCTS_TARGET_TABLE
    )
)

products_query.awaitTermination()

print(
    "Company A products Bronze ingestion completed."
)

**Validate the Products**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_product_count
FROM workspace.bronze.company_a_products
""").show()

In [0]:
spark.sql("""
SELECT
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_a_products
GROUP BY _source_file
""").show(truncate=False)

### Ingesting Orders data into Bronze

**Setting the path**

In [0]:
ORDERS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_a/raw/orders/"
)

ORDERS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_a/orders/"
)

ORDERS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_a/orders/"
)

ORDERS_TARGET_TABLE = (
    "workspace.bronze.company_a_orders"
)

**Reading with Autoloader**

In [0]:
orders_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        ORDERS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(ORDERS_SOURCE_PATH)
)

**Adding the metadata**

In [0]:
orders_bronze_df = (
    orders_raw_df
    .withColumn(
        "_source_system",
        lit("company_a"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Orders Data to Bronze Delta table**

In [0]:
orders_query = (
    orders_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        ORDERS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        ORDERS_TARGET_TABLE
    )
)

orders_query.awaitTermination()

print(
    "Company A orders Bronze ingestion completed."
)

**Validate**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_order_count
FROM workspace.bronze.company_a_orders
""").show()

In [0]:
spark.sql("""
SELECT
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_a_orders
GROUP BY _source_file
ORDER BY _source_file
""").show(
    30,
    truncate=False,
)

###  Ingesting Orders_Items data into Bronze

**Setting the path**

In [0]:
ORDER_ITEMS_SOURCE_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "company_a/raw/order_items/"
)

ORDER_ITEMS_SCHEMA_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_autoloader_schema/company_a/order_items/"
)

ORDER_ITEMS_CHECKPOINT_PATH = (
    "s3://rahul-company-merger-data-lake/"
    "_checkpoints/company_a/order_items/"
)

ORDER_ITEMS_TARGET_TABLE = (
    "workspace.bronze.company_a_order_items"
)

**Reading with Autoloader**

In [0]:
order_items_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        ORDER_ITEMS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "false",
    )
    .load(ORDER_ITEMS_SOURCE_PATH)
)

**Adding the metadata**

In [0]:
order_items_bronze_df = (
    order_items_raw_df
    .withColumn(
        "_source_system",
        lit("company_a"),
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name"),
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path"),
    )
    .withColumn(
        "_file_modification_time",
        col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        current_timestamp(),
    )
)

**Writing Orders_items Data to Bronze Delta Table**

In [0]:
order_items_query = (
    order_items_bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        ORDER_ITEMS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        ORDER_ITEMS_TARGET_TABLE
    )
)

order_items_query.awaitTermination()

print(
    "Company A order items Bronze ingestion completed."
)

**Validate**

In [0]:
spark.sql("""
SELECT COUNT(*) AS bronze_order_item_count
FROM workspace.bronze.company_a_order_items
""").show()

In [0]:
spark.sql("""
SELECT
    _source_file,
    COUNT(*) AS row_count
FROM workspace.bronze.company_a_order_items
GROUP BY _source_file
ORDER BY _source_file
""").show(
    30,
    truncate=False,
)